# NER hard cases demo

This notebook demonstrates how different NER systems behave on a small set of deliberately difficult examples.

It includes:
- boundary ambiguity cases
- ambiguity / polysemy
- noisy text
- unseen / rare entities
- metonymy
- nested-entity-like contexts that standard flat NER often struggles with

The notebook compares:
- **spaCy**
- **Hugging Face transformers pipeline**
- **Flair** (optional)

It computes:
- **exact-span matching**
- **relaxed overlap matching**
- per-example inspection tables

> The examples are illustrative and hand-annotated for demonstration purposes.


In [ ]:
# Optional installs (run only if needed)
!pip install -q spacy transformers torch pandas seqeval flair
!python -m spacy download en_core_web_sm


In [4]:
from __future__ import annotations

import re
import math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Any
from collections import defaultdict

import pandas as pd


In [ ]:
# -----------------------------
# Example set
# -----------------------------

examples = [
    {
        "id": "boundary_1",
        "category": "boundary_ambiguity",
        "text": "He works at Apple Inc. headquarters in Cupertino.",
        "gold": [
            {"text": "Apple Inc.", "label": "ORG"},
            {"text": "Cupertino", "label": "LOC"},
        ],
        "note": "Boundary ambiguity: a model may wrongly extend ORG to 'Apple Inc. headquarters'."
    },
    {
        "id": "boundary_2",
        "category": "boundary_ambiguity",
        "text": "President Barack Obama gave a speech in Berlin.",
        "gold": [
            {"text": "Barack Obama", "label": "PER"},
            {"text": "Berlin", "label": "LOC"},
        ],
        "note": "Boundary ambiguity: some schemes exclude the title 'President'."
    },
    {
        "id": "ambiguity_1",
        "category": "polysemy",
        "text": "I flew into JFK yesterday.",
        "gold": [
            {"text": "JFK", "label": "LOC"},
        ],
        "note": "Ambiguous surface form: JFK may refer to a person or an airport."
    },
    {
        "id": "ambiguity_2",
        "category": "polysemy",
        "text": "JFK was assassinated in 1963.",
        "gold": [
            {"text": "JFK", "label": "PER"},
        ],
        "note": "Same token, different type from the previous example."
    },
    {
        "id": "metonymy_1",
        "category": "metonymy",
        "text": "The White House announced new policies today.",
        "gold": [
            {"text": "The White House", "label": "ORG"},
        ],
        "note": "Metonymy: literal place name used to refer to an organization."
    },
    {
        "id": "noise_1",
        "category": "noisy_text",
        "text": "met w/ @elonmusk in nyc lol",
        "gold": [
            {"text": "@elonmusk", "label": "PER"},
            {"text": "nyc", "label": "LOC"},
        ],
        "note": "No capitalization, social media style, handle-based mention."
    },
    {
        "id": "rare_1",
        "category": "rare_unseen",
        "text": "DeepGenX Labs raised funding in Aarhus.",
        "gold": [
            {"text": "DeepGenX Labs", "label": "ORG"},
            {"text": "Aarhus", "label": "LOC"},
        ],
        "note": "Rare / invented organization name."
    },
    {
        "id": "biomed_1",
        "category": "domain_specific",
        "text": "BRCA1 mutation was detected at Aalborg University Hospital.",
        "gold": [
            {"text": "BRCA1", "label": "GENE"},
            {"text": "Aalborg University Hospital", "label": "ORG"},
        ],
        "note": "Domain-specific entity plus a multi-token organization."
    },
    {
        "id": "boundary_3",
        "category": "boundary_ambiguity",
        "text": "She studied at the University of Oxford in England.",
        "gold": [
            {"text": "University of Oxford", "label": "ORG"},
            {"text": "England", "label": "LOC"},
        ],
        "note": "ORG boundary may be wrongly extended to include trailing prepositional phrase."
    },
]
len(examples)


In [7]:
# -----------------------------
# Helper functions
# -----------------------------

def find_span(text: str, phrase: str, start_hint: int = 0) -> Tuple[int, int]:
    start = text.find(phrase, start_hint)
    if start == -1:
        raise ValueError(f"Could not find phrase {phrase!r} in text {text!r}")
    return start, start + len(phrase)

def prepare_gold(examples: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    prepared = []
    for ex in examples:
        text = ex["text"]
        gold_spans = []
        cursor = 0
        for ent in ex["gold"]:
            s, e = find_span(text, ent["text"], cursor)
            gold_spans.append({
                "start": s,
                "end": e,
                "text": text[s:e],
                "label": ent["label"],
            })
            cursor = s + 1
        prepared.append({**ex, "gold_spans": gold_spans})
    return prepared

examples = prepare_gold(examples)

def normalize_label(label: str, system: str = "") -> str:
    label = label.upper()
    mapping = {
        "PERSON": "PER",
        "PER": "PER",
        "ORG": "ORG",
        "ORGANIZATION": "ORG",
        "LOC": "LOC",
        "LOCATION": "LOC",
        "GPE": "LOC",
        "FAC": "LOC",
        "GENE": "GENE",
    }
    return mapping.get(label, label)

def overlaps(a: Dict[str, Any], b: Dict[str, Any]) -> bool:
    return max(a["start"], b["start"]) < min(a["end"], b["end"])

def exact_match(a: Dict[str, Any], b: Dict[str, Any]) -> bool:
    return a["start"] == b["start"] and a["end"] == b["end"] and a["label"] == b["label"]

def relaxed_match(a: Dict[str, Any], b: Dict[str, Any]) -> bool:
    return overlaps(a, b) and a["label"] == b["label"]

def score_one(gold: List[Dict[str, Any]], pred: List[Dict[str, Any]], relaxed: bool = False) -> Dict[str, int]:
    matcher = relaxed_match if relaxed else exact_match
    used = set()
    tp = 0
    for g_idx, g in enumerate(gold):
        for p_idx, p in enumerate(pred):
            if p_idx in used:
                continue
            if matcher(g, p):
                tp += 1
                used.add(p_idx)
                break
    fp = len(pred) - tp
    fn = len(gold) - tp
    return {"tp": tp, "fp": fp, "fn": fn}

def prf(tp: int, fp: int, fn: int) -> Dict[str, float]:
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2*p*r/(p+r) if p+r else 0.0
    return {"precision": p, "recall": r, "f1": f1}

def aggregate(scores: List[Dict[str, int]]) -> Dict[str, float]:
    tp = sum(x["tp"] for x in scores)
    fp = sum(x["fp"] for x in scores)
    fn = sum(x["fn"] for x in scores)
    return {"tp": tp, "fp": fp, "fn": fn, **prf(tp, fp, fn)}

def ents_to_df(ents: List[Dict[str, Any]]) -> pd.DataFrame:
    if not ents:
        return pd.DataFrame(columns=["start", "end", "text", "label"])
    return pd.DataFrame(ents)[["start", "end", "text", "label"]]


In [8]:
# Show the gold annotations
for ex in examples:
    print(f"\n[{ex['id']}] {ex['category']}")
    print(ex["text"])
    display(ents_to_df(ex["gold_spans"]))
    print("Note:", ex["note"])



[boundary_1] boundary_ambiguity
He works at Apple Inc. headquarters in Cupertino.


,start,end,text,label
0,12,22,Apple Inc.,ORG
1,39,48,Cupertino,LOC


Note: Boundary ambiguity: a model may wrongly extend ORG to 'Apple Inc. headquarters'.

[boundary_2] boundary_ambiguity
President Barack Obama gave a speech in Berlin.


,start,end,text,label
0,10,22,Barack Obama,PER
1,40,46,Berlin,LOC


Note: Boundary ambiguity: some schemes exclude the title 'President'.

[ambiguity_1] polysemy
I flew into JFK yesterday.


...

Note: Rare / invented organization name.

[biomed_1] domain_specific
BRCA1 mutation was detected at Aalborg University Hospital.


,start,end,text,label
0,0,5,BRCA1,GENE
1,31,58,Aalborg University Hospital,ORG


Note: Domain-specific entity plus a multi-token organization.

[boundary_3] boundary_ambiguity
She studied at the University of Oxford in England.


,start,end,text,label
0,19,39,University of Oxford,ORG
1,43,50,England,LOC


Note: ORG boundary may be wrongly extended to include trailing prepositional phrase.


In [9]:
# -----------------------------
# Model wrappers
# -----------------------------

def run_spacy(texts: List[str], model_name: str = "en_core_web_sm") -> List[List[Dict[str, Any]]]:
    import spacy
    nlp = spacy.load(model_name)
    outputs = []
    for doc in nlp.pipe(texts):
        ents = []
        for ent in doc.ents:
            ents.append({
                "start": ent.start_char,
                "end": ent.end_char,
                "text": ent.text,
                "label": normalize_label(ent.label_, "spacy"),
            })
        outputs.append(ents)
    return outputs

def run_hf(texts: List[str], model_name: str = "dslim/bert-base-NER") -> List[List[Dict[str, Any]]]:
    from transformers import pipeline
    pipe = pipeline("token-classification", model=model_name, aggregation_strategy="simple")
    outputs = []
    for text in texts:
        raw = pipe(text)
        ents = []
        for ent in raw:
            ents.append({
                "start": int(ent["start"]),
                "end": int(ent["end"]),
                "text": text[int(ent["start"]):int(ent["end"])],
                "label": normalize_label(ent["entity_group"], "hf"),
            })
        outputs.append(ents)
    return outputs

def run_flair(texts: List[str], model_name: str = "ner") -> List[List[Dict[str, Any]]]:
    from flair.data import Sentence
    from flair.nn import Classifier
    tagger = Classifier.load(model_name)
    outputs = []
    for text in texts:
        sentence = Sentence(text)
        tagger.predict(sentence)
        ents = []
        for span in sentence.get_spans("ner"):
            ents.append({
                "start": span.start_position,
                "end": span.end_position,
                "text": text[span.start_position:span.end_position],
                "label": normalize_label(span.get_label("ner").value, "flair"),
            })
        outputs.append(ents)
    return outputs


In [10]:
# -----------------------------
# Run selected models
# -----------------------------

texts = [ex["text"] for ex in examples]

predictions = {}

# spaCy
try:
    predictions["spaCy"] = run_spacy(texts)
    print("spaCy loaded successfully.")
except Exception as e:
    print("spaCy failed:", e)

# Hugging Face
try:
    predictions["HF_dslim_bert_base_NER"] = run_hf(texts)
    print("Hugging Face pipeline loaded successfully.")
except Exception as e:
    print("Hugging Face pipeline failed:", e)

# Flair
try:
    predictions["Flair_ner"] = run_flair(texts)
    print("Flair loaded successfully.")
except Exception as e:
    print("Flair failed:", e)

print("Loaded systems:", list(predictions.keys()))


spaCy loaded successfully.


Hugging Face pipeline loaded successfully.


Flair loaded successfully.
Loaded systems: ['spaCy', 'HF_dslim_bert_base_NER', 'Flair_ner']


In [ ]:
# -----------------------------
# Evaluate
# -----------------------------

rows = []

for system_name, system_preds in predictions.items():
    exact_scores = []
    relaxed_scores = []
    for ex, pred in zip(examples, system_preds):
        exact_scores.append(score_one(ex["gold_spans"], pred, relaxed=False))
        relaxed_scores.append(score_one(ex["gold_spans"], pred, relaxed=True))
    exact = aggregate(exact_scores)
    relaxed = aggregate(relaxed_scores)
    rows.append({
        "system": system_name,
        "metric": "exact",
        **exact,
    })
    rows.append({
        "system": system_name,
        "metric": "relaxed_overlap",
        **relaxed,
    })

summary_df = pd.DataFrame(rows)
summary_df.sort_values(["metric", "f1"], ascending=[True, False])


In [ ]:
# -----------------------------
# Per-example breakdown
# -----------------------------

details = []

for system_name, system_preds in predictions.items():
    for ex, pred in zip(examples, system_preds):
        exact = score_one(ex["gold_spans"], pred, relaxed=False)
        relaxed = score_one(ex["gold_spans"], pred, relaxed=True)
        details.append({
            "system": system_name,
            "id": ex["id"],
            "category": ex["category"],
            "text": ex["text"],
            "exact_f1": prf(**exact)["f1"],
            "relaxed_f1": prf(**relaxed)["f1"],
            "gold_n": len(ex["gold_spans"]),
            "pred_n": len(pred),
            "note": ex["note"],
        })

details_df = pd.DataFrame(details)
details_df.sort_values(["category", "id", "system"])


In [ ]:
# -----------------------------
# Category-wise comparison
# -----------------------------

category_rows = []

for system_name, system_preds in predictions.items():
    by_cat_exact = defaultdict(list)
    by_cat_relaxed = defaultdict(list)
    for ex, pred in zip(examples, system_preds):
        by_cat_exact[ex["category"]].append(score_one(ex["gold_spans"], pred, relaxed=False))
        by_cat_relaxed[ex["category"]].append(score_one(ex["gold_spans"], pred, relaxed=True))
    for cat in sorted(by_cat_exact):
        exact = aggregate(by_cat_exact[cat])
        relaxed = aggregate(by_cat_relaxed[cat])
        category_rows.append({"system": system_name, "category": cat, "metric": "exact", **exact})
        category_rows.append({"system": system_name, "category": cat, "metric": "relaxed_overlap", **relaxed})

category_df = pd.DataFrame(category_rows)
category_df.sort_values(["category", "metric", "f1"], ascending=[True, True, False])


In [ ]:
# -----------------------------
# Inspect one example in detail
# -----------------------------

example_id = "boundary_1"  # change this

ex = next(x for x in examples if x["id"] == example_id)
print(f"[{ex['id']}] {ex['category']}")
print(ex["text"])
print("Note:", ex["note"])
print("\nGold:")
display(ents_to_df(ex["gold_spans"]))

for system_name, system_preds in predictions.items():
    idx = [i for i, x in enumerate(examples) if x["id"] == example_id][0]
    print(f"\nPredictions from {system_name}:")
    display(ents_to_df(system_preds[idx]))


In [ ]:
# -----------------------------
# Simple baseline: regex / gazetteer-like toy recognizer
# This is intentionally weak, but sometimes useful for teaching.
# -----------------------------

KNOWN = {
    "Apple Inc.": "ORG",
    "Cupertino": "LOC",
    "Barack Obama": "PER",
    "Berlin": "LOC",
    "JFK": "PER",  # intentionally problematic baseline
    "The White House": "LOC",  # intentionally literal, not metonymic
    "@elonmusk": "PER",
    "nyc": "LOC",
    "Aarhus": "LOC",
    "Aalborg University Hospital": "ORG",
    "University of Oxford": "ORG",
    "England": "LOC",
    "BRCA1": "GENE",
}

def run_toy_baseline(texts: List[str]) -> List[List[Dict[str, Any]]]:
    outputs = []
    for text in texts:
        ents = []
        for phrase, label in KNOWN.items():
            start = text.find(phrase)
            if start != -1:
                ents.append({
                    "start": start,
                    "end": start + len(phrase),
                    "text": phrase,
                    "label": label,
                })
        ents = sorted(ents, key=lambda x: (x["start"], x["end"]))
        outputs.append(ents)
    return outputs

predictions["Toy_baseline"] = run_toy_baseline(texts)
print("Toy baseline added.")


In [ ]:
# Recompute summary including toy baseline

rows = []
for system_name, system_preds in predictions.items():
    exact_scores = []
    relaxed_scores = []
    for ex, pred in zip(examples, system_preds):
        exact_scores.append(score_one(ex["gold_spans"], pred, relaxed=False))
        relaxed_scores.append(score_one(ex["gold_spans"], pred, relaxed=True))
    rows.append({"system": system_name, "metric": "exact", **aggregate(exact_scores)})
    rows.append({"system": system_name, "metric": "relaxed_overlap", **aggregate(relaxed_scores)})

pd.DataFrame(rows).sort_values(["metric", "f1"], ascending=[True, False])


## Notes for teaching

A few useful discussion points:
- **Exact match** is harsh: a semantically reasonable entity with slightly wrong boundaries still gets no credit.
- **Relaxed overlap** often reveals that a model “roughly knew” where the entity was.
- Flat NER models often struggle with:
  - **titles and trailing descriptors**
  - **metonymy**
  - **social-media noise**
  - **rare or invented names**
  - **domain-specific labels** like `GENE`, which many general-purpose models do not support
- The toy baseline can sometimes beat neural models on memorized entities, while failing badly on ambiguity.


In [ ]:
# -----------------------------
# Detailed results per model
# -----------------------------

for system_name, system_preds in predictions.items():
    print("=" * 70)
    print(f"MODEL: {system_name}")
    print("=" * 70)
    for ex, pred in zip(examples, system_preds):
        exact  = score_one(ex["gold_spans"], pred, relaxed=False)
        relaxed = score_one(ex["gold_spans"], pred, relaxed=True)
        exact_f1   = prf(**exact)["f1"]
        relaxed_f1 = prf(**relaxed)["f1"]
        print(f"\n  [{ex['id']}] {ex['category']}")
        print(f"  Text : {ex['text']}")
        print(f"  Note : {ex['note']}")
        print(f"  Exact F1 : {exact_f1:.2f}   Relaxed F1 : {relaxed_f1:.2f}")
        print("  Gold entities:")
        display(ents_to_df(ex["gold_spans"]))
        print("  Predicted entities:")
        display(ents_to_df(pred))
    print()
